In [1]:
import pdal, json
import numpy as np
import trimesh

# === entrada e saída ===
entrada = "../downloads/MDS_3313-144_1000.laz"
saida_glb = "resultados/MDS_3313-144_cov16.glb"

# === análise de sensibilidade (kNN) ===
# Teste com diferentes valores de kNN para ver o impacto nas features de covariância
# kNN = 1, 4, 8, 16, 32, 64
# Valores maiores de kNN capturam mais contexto local, mas podem suavizar detalhes finos.
# kNN = 16 é um bom compromisso para muitos casos.
kNN = 16

# === pipeline PDAL ===
pipeline_json = {
    "pipeline": [
        {"type": "readers.las", "filename": entrada},
        # {"type": "filters.covariancefeatures", "knn": kNN}
    ]
}

# executa
p = pdal.Pipeline(json.dumps(pipeline_json))
p.execute()

# obtém a nuvem processada (com features)
arr = p.arrays[0]

# === coordenadas XYZ ===
xyz = np.vstack((arr["X"], arr["Y"], arr["Z"])).T.astype(np.float32)

# === centraliza em (0,0,0) ===
centroid = xyz.mean(axis=0)
xyz_centered = xyz - centroid

# === aplica rotação de -90° em X (Z-up → Y-up) ===
R = trimesh.transformations.rotation_matrix(np.radians(-90), [1, 0, 0])
xyz_rot = trimesh.transform_points(xyz_centered, R)

# === cria cores com base na intensidade (tons de cinza) ===
if "Intensity" in arr.dtype.names:
    intensity = arr["Intensity"].astype(np.float32)
    norm_intensity = (intensity - intensity.min()) / (intensity.ptp() + 1e-9)
    colors = np.stack([norm_intensity]*3, axis=1)
    colors = (colors * 255).astype(np.uint8)
else:
    colors = np.ones_like(xyz_rot) * 255  # branco

# === cria PointCloud ===
cloud = trimesh.points.PointCloud(vertices=xyz_rot, colors=colors)

# === 🔹 salva o arquivo GLB ===
cloud.export(saida_glb)  # <-- ESTA É A LINHA QUE GRAVA O ARQUIVO

print(f"✅ Nuvem processada: {len(xyz_rot):,} pontos")
print(f"🔹 Centro original: {centroid.round(3)}")
print(f"📦 Exportado para: {saida_glb}")



✅ Nuvem processada: 6,970,391 pontos
🔹 Centro original: [3.2217203e+05 7.3934805e+06 7.4514801e+02]
📦 Exportado para: resultados/MDS_3313-144_cov16.glb
